# Prerequisite · Topic 1 — Jupyter (and VS Code notebooks)

---

## 1. What it is

**Jupyter** is an interactive computing environment. Instead of running a whole program top-to-bottom, you write small **cells** of code, run them one at a time, and see each output immediately underneath. The working document is a **notebook** (`.ipynb`) that holds code cells, their outputs (numbers, tables, plots), and Markdown prose, interleaved.

A notebook is three things fused into one document: **code + its results + your narrative**.

## 2. Why it exists

- **Exploratory, iterative analysis.** Single-cell work is decide-as-you-look: you run a step, inspect the result (a QC plot, a surviving-cell count, a cluster), then choose the next step. A plain `.py` script re-runs everything each time — terrible for this.
- **State stays alive in memory.** Load the data once (slow), then run many quick experiments on it (fast) without reloading.
- **Literate programming.** Code + output + explanation live together, so the notebook doubles as a lab notebook and a shareable report — future-you understands *what* and *why*.

## 3. When it is used

For **exploration, prototyping, teaching, and narrated analyses** — i.e. this whole project. (Reusable code you import and run repeatedly eventually belongs in plain `.py` modules, but that's a later concern.)

## 4. How it works internally — the document vs the kernel

Two separate things; confusing them causes most beginner pain.

```
   YOU (VS Code / JupyterLab)          THE KERNEL
   ┌───────────────────────┐          ┌──────────────────────────┐
   │ notebook document     │  send    │ a live Python process     │
   │  = .ipynb (JSON file) │ ───────▶ │  that runs your code and  │
   │  cells + saved outputs │          │  HOLDS ALL VARIABLES      │
   │                        │ ◀─────── │  in memory                │
   └───────────────────────┘  results └──────────────────────────┘
        the document                        the "brain"
```

- **Notebook document** (`.ipynb`) — a text file (JSON) storing cells and last-saved outputs. Holds no memory, runs nothing.
- **Kernel** — a *separate, live Python process* that executes cells and remembers every variable. **State lives in the kernel, never in the document.** Close the kernel and all variables/data are gone, even though the file still shows old outputs.

Two consequences to internalize:

1. **State follows execution order, not cell position.** Each run cell shows a counter `[1]`, `[2]`, `[3]` = the order you *ran* them. Running bottom-then-top makes the kernel reflect *that* sequence, regardless of top-to-bottom reading order.
2. **The kernel is a specific environment's Python.** If the kernel is `base`, it can't see packages installed into `scrna`. **Choosing the kernel = choosing which environment's Python runs your code** — the same PATH/activation idea from Topic 1.

## 5. Common mistakes

- **Hidden / out-of-order state (cardinal sin).** Editing and re-running cells in a jumble so results depend on a run order the document no longer matches → reopen and run top-to-bottom and it breaks or gives different numbers. Destroys reproducibility silently.
- **Wrong kernel** → `ModuleNotFoundError` for a package you *know* you installed. Cause: kernel isn't the `scrna` env. (VS Code is especially prone: it defaults to whatever interpreter it found.)
- **Trusting a long-lived kernel** where a variable from a since-deleted cell keeps things "working" — until a restart proves it doesn't.
- **Monster notebooks** — hundreds of cells, huge embedded outputs, no structure.

## 6. Best practices

- **Install Jupyter/ipykernel into `scrna`** so the kernel is guaranteed to be the project Python.
- **"Restart Kernel and Run All" regularly**, and always before trusting or sharing results. Clean top-to-bottom run = honest notebook. This single habit neutralizes out-of-order state.
- **Small, one-idea cells**, logically grouped, with Markdown headers.
- **Prove the kernel** with `sys.executable` instead of assuming.

## 7. Alternatives

| Option | Note |
|---|---|
| **JupyterLab** | Modern Jupyter interface (files + notebooks + terminal). |
| **VS Code notebooks** | Run `.ipynb` inside VS Code; same kernel concepts. Pick the `scrna` interpreter as kernel. |
| Classic Notebook | Older, simpler; Lab is the current standard. |
| Google Colab | Cloud, zero setup — but you lose control of the conda env; poor fit for a reproducibility-focused project. |

Either JupyterLab or VS Code is fine here because both run locally against the conda env you control.

## 8. Hands-on

**Purpose:** install the interface + kernel bridge into `scrna`, then *prove* the notebook runs the project Python.

```bash
# Always start in the project env
conda activate scrna

# Install the interface + ipykernel (lets this env act as a kernel)
conda install -c conda-forge jupyterlab ipykernel -y
```

- Installing *inside* `scrna` is what guarantees its kernel is our Python; installing in `base` is the #1 cause of the wrong-kernel bug.
- `jupyterlab` = interface; `ipykernel` = the bridge that makes a conda env show up as a runnable kernel. Need both.

**Path A — JupyterLab:**
```bash
jupyter lab        # opens in the browser; new notebook offers the scrna kernel
```

**Path B — VS Code (default editor here):**
1. Install the **Python** and **Jupyter** extensions.
2. Open or create a `.ipynb` file.
3. Click the **kernel selector** (top-right) → choose the **`scrna`** environment.
4. If `scrna` isn't listed, register it once:
   ```bash
   python -m ipykernel install --user --name scrna
   ```

**Proof cell (run first, in either path):**


In [2]:
import sys
print(sys.executable)     # MUST contain .../envs/scrna/...  — proves the kernel
 
import numpy, pandas, anndata
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("anndata:", anndata.__version__)

/home/amirhossein/miniconda3/envs/SingleCell/bin/python
numpy: 2.4.6
pandas: 3.0.3
anndata: 0.13.0


/tmp/ipykernel_9166/725481969.py:7: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", anndata.__version__)


- `sys.executable` prints the full path of the running Python. It must include `.../envs/scrna/...`. Anything else = wrong kernel; switch before continuing.
- The three imports succeeding proves the kernel sees the Topic 1 stack. An import failure here is the wrong-kernel problem, not a broken package.
## 9. Troubleshooting: `ModuleNotFoundError` for a package you installed
 
Cause: the kernel is a *different* Python than the env you installed into — Python only sees packages in the interpreter that's running. Fix: switch the kernel to `scrna` and confirm via `sys.executable`. If the env isn't offered as a kernel, register it with `python -m ipykernel install --user --name scrna`.
 
## 10. One-line summary
 
Jupyter runs code in small cells with inline output, mixing code + results + prose in an `.ipynb`; the document only stores cells and saved outputs, while a separate **kernel** (a specific conda env's Python) runs the code and holds all state — so pick the `scrna` kernel, verify with `sys.executable`, and "Restart & Run All" to keep the notebook honest.